# Latent Dirichlet Allocation

Latent Dirichlet Allocation (LDA) represents documents as combinations of topics for which each word has a certain probability. The statistics underlying this approach to text analysis assumes a generative probabilistic model for any set of discrete data (in this case, text documents). In other words, LDA relies on the assumption that each topic may be modeled as an "infinite mixture over an underlying set of topic probabilities" [1]. The number of words in a document follows a certain distribution and the topic composition of the document follows a Dirichlet distribution over a fixed set of topics. 
<br>
<img src="http://deliveryimages.acm.org/10.1145/1860000/1859210/figs/uf1.jpg" align="center" height="500" width="500">
<br>
The purpose of LDA is to determine a set of topics that are likely to have generated the documents from the words in the documents themselves. [2] One shortcoming of the LDA model is it uses the "bag of words" assumption that the order of words in a document may be ignored and that, therefore, documents are more or less exchangable. Non-probabilistic instantiations of LDA mainly use word counts as features.

<br>
[1] D. M. Blei, A. Y. Ng, M. I. Jordan, 2003. "Latent Dirichlet Allocation." *Journal of Machine Learning Research*. pp. 993-1022. [Online](https://www.cs.princeton.edu/~blei/papers/BleiNgJordan2003.pdf).<br>
[2] E. Chen, 2013. "Introduction to Latent Dirichlet Allocation." [Online](http://blog.echen.me/2011/08/22/introduction-to-latent-dirichlet-allocation/). <br>
[3] G. Anthes, 2010. "Topic Models vs. Unstructured Data." *Communications of the ACM* 53(12): p. 16-18. DOI:
10.1145/1859204.1859210. 

In [ ]:
%matplotlib inline
import numpy as np
import scipy as sp
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import pandas as pd
import time
pd.set_option("display.width", 500)
pd.set_option("display.max_columns", 100)
pd.set_option("display.notebook_repr_html", True)
import seaborn as sns
sns.set_style("darkgrid")
sns.set_context("poster")

### Import and Clean Data

Using the methods defined in the Data Cleaning notebook, we can transform the text into training and test corpuses that are segmented by word type (noun, adjective, verb, foreign, and precedent).

In [ ]:
sample_df = pd.read_csv("sample_cases.csv")

In [ ]:
sample_df.head()

In [ ]:
# training, test data split 
trainingcoln = pd.read_csv('traintestarray.csv',sep=',',header=None).values.ravel()
sample_df['training'] = trainingcoln

The following cleaning functions can also be found in the Data Cleaning notebook.

In [ ]:
import re 
regex1 = r"\(.\)" 

In [ ]:
from pattern.en import parse
from pattern.en import pprint
from pattern.en import conjugate, lemma, lexeme
from pattern.vector import stem, PORTER, LEMMA
from sklearn.feature_extraction import text
import string

#stopwords and punctuation
stopwords=text.ENGLISH_STOP_WORDS
punctuation = list('.,;:!?()[]{}`''\"@#$^&*+-|=~_')

def get_parts(opinion):
    oplow = opinion.lower()
    #REMOVING CHARACTERS: we have ugly text, and remove unnecssary characters.
    oplow = unicode(oplow, 'ascii', 'ignore') #remove non-unicode characters 
    oplow = str(oplow).translate(string.maketrans("\n\t\r", "   ")) #remove characters like \n 
    #justices (eg, Justice Breyer) are referred to as J. (eg,Breyer, J.); we remove the J., also JJ. for plural
    oplow = oplow.replace('j.','')
    oplow = oplow.replace('jj.','')
    oplow = oplow.replace('c.','') #remove C. for chief justice 
    oplow = oplow.replace('pp.','') #page numbers
    oplow = oplow.replace('  ','') #multiple spaces
    oplow = ''.join([i for i in oplow if not i.isdigit()]) #remove digits 
    oplow=re.sub(regex1, ' ', oplow)
    #Remove the Justia disclaimer at the end of the case, if it appears in the string
    justiadisclaimer = "disclaimer: official"
    if justiadisclaimer in oplow: 
        optouse = oplow.split(justiadisclaimer)[0]
    else:
        optouse = oplow
    
    #GET A LIST OF PRECEDENTS CITED IN THE OPINION 
    wordslist = optouse.split()
    #find precedents based on string 'v.' (eg, 'Brown v. Board')
    indices = [i for i in range(len(wordslist)) if wordslist[i]=='v.']
    precedents = [wordslist[i-1]+ ' ' + wordslist[i]+ ' ' + wordslist[i+1] for i in indices]
    
    #remove precedents, as we have already accounted for these
    for precedent in precedents:
        optouse = optouse.replace(precedent,'')
    
    #PARSE INTO LIST OF LISTS --> GET WORDS
    parsed = parse(optouse,tokenize=True,chunks=False,lemmata=True).split()
    verbs = [] 
    nouns = [] 
    adjectives = [] 
    foreign = [] 
    i=0
    #Create lists of lists of verbs, nouns, adjectives and foreign words in each sentence.
    for sentence in parsed: #for each sentence 
        verbs.append([])
        nouns.append([])
        adjectives.append([])
        foreign.append([])
        for token in sentence: #for each word in the sentence 
            if token[0] in punctuation or token[0] in stopwords or len(token[0])<=2:
                continue
            wordtouse = token[0]
            for x in punctuation:
                wordtouse = wordtouse.replace(x,' ') #if punctuation in word, take it out
            if token[1] in ['VB','VBZ','VBP','VBD','VBN','VBG']:
                verbs[i].append(lemma(wordtouse)) #append the lemmatized verb (we relemmatize because lemmata in parse does not seem to always work)
            if token[1] in ['NN','NNS','NNP','NNPS']:
                nouns[i].append(lemma(wordtouse))
            if token[1] in ['JJ','JJR','JJS']:
                adjectives.append(lemma(wordtouse))
            if token[1] in ['FW']:
                foreign.append(wordtouse)  
        i+=1  
    #Zip together lists so each tuple is a sentence. 
    out=zip(verbs,nouns,adjectives,foreign)
    verbs2 = []
    nouns2 = []
    adjectives2 = []
    foreign2 = []
    for sentence in out: 
        if sentence[0]!=[] and sentence[1]!=0: #if the sentence has at least one verb and noun, keep it. Otherwise, drop it.
            if type(sentence[0])==list: 
                verbs2.append(sentence[0])
            else: 
                verbs2.append([sentence[0]]) #if verb is a string rather than a list, put string in list
            if type(sentence[1])==list:
                nouns2.append(sentence[1])
            else:
                nouns2.append([sentence[1]])
            if type(sentence[2])==list:
                adjectives2.append(sentence[2])
            else:
                adjectives2.append([sentence[2]])
            if type(sentence[3])==list:
                foreign2.append(sentence[3])
            else:
                foreign2.append([sentence[3]])
    return(verbs2,nouns2,adjectives2,foreign2,precedents)

### Vocabulary Creation

Now that we have cleaned our text corpus, we want to create vocabularies segmented by word type. We use the **get_parts** function defined above to return all verbs, nouns, adjectives, foreign words, and precedents. This word list will be useful for creating matrices of word frequencies. 

In [ ]:
%%time 
verbwords = []
nounwords = []
adjwords = []
forwords = []
precedents_all = []
for op in sample_df.text:
    verbs,nouns,adjectives,foreign,precedents = get_parts(op)
    verbwords.append(verbs)
    nounwords.append(nouns)
    adjwords.append(adjectives)
    forwords.append(foreign)
    precedents_all.append(precedents)

In [ ]:
issue_areas = sample_df.issueArea.tolist()

In [ ]:
#create precedents vocab
precedents_vocab = list(set([precedent for sublist in precedents_all for precedent in sublist]))
#create other vocabs
verbvocab = list(set([word for sublist in verbwords for subsublist in sublist for word in subsublist]))
nounvocab = list(set([word for sublist in nounwords for subsublist in sublist for word in subsublist]))
adjvocab = list(set([word for sublist in adjwords for subsublist in sublist for word in subsublist]))
forvocab = list(set([word for sublist in forwords for subsublist in sublist for word in subsublist]))

In [ ]:
#dictionaries: id --> word
id2prec = dict(enumerate(precedents_vocab))
id2verb = dict(enumerate(verbvocab))
id2noun = dict(enumerate(nounvocab))
id2adj = dict(enumerate(adjvocab))
id2for = dict(enumerate(forvocab))
#dictionaries: word --> id
prec2id = dict(zip(id2prec.values(),id2prec.keys()))
verb2id = dict(zip(id2verb.values(),id2verb.keys()))
noun2id = dict(zip(id2noun.values(),id2noun.keys()))
adj2id = dict(zip(id2adj.values(),id2adj.keys()))
for2id = dict(zip(id2for.values(),id2for.keys()))

In [ ]:
#this function takes a list of words, and outputs a list of tuples 
counter = lambda x:list(set([(i,x.count(i)) for i in x]))

#corpus_creator takes a list of lists of lists like verbwords, or a list of lists like precedents_all. 
#It also takes a word2id dictionary.
def corpus_creator(sentence_word_list,word2id):
    counter = lambda x:list(set([(word2id[i],x.count(i)) for i in x]))
    op_word_list = []
    if type(sentence_word_list[0][0])==list: #if list of lists of lists 
        for opinion in sentence_word_list: 
            #for each list (which corresponds to an opinion) in sentence_word_list, get a list of the words
            op_word_list.append([word for sublist in opinion for word in sublist])
    else: #if list of lists 
        op_word_list = sentence_word_list
    corpus = []
    for element in op_word_list: 
        corpus.append(counter(element))
    return(corpus)

### First Pass (Untransformed)

To adjust the parameters of our mode, we will use the corpus of nouns to check for model accuracy since nouns are most strongly indicative of topics. The first pass at LDA will use the untransformed word matrices created above. We will adjust the number of topic categories in order to finetune the model output.

In [ ]:
import gensim

In [ ]:
corpus = corpus_creator(nounwords,noun2id)

In [ ]:
%%time 
# model with noun corpus, 5 topics
lda1a = gensim.models.ldamodel.LdaModel(corpus=corpus, id2word=id2noun, num_topics=5, update_every=1, chunksize=200, passes=1)

In [ ]:
lda1a.print_topics()

In [ ]:
%%time 
# model with noun corpus, 10 topics
lda1b = gensim.models.ldamodel.LdaModel(corpus=corpus, id2word=id2noun, num_topics=10, update_every=1, chunksize=200, passes=1)

In [ ]:
lda1b.print_topics()

### Second Pass (td-idf)

We can see from the words and corresponding coefficients above that there are some frequent words that occur in each topic (i.e. court, case, action, opinion, respondent). In order to refine the words that are generated by each topic, we can use term frequency?inverse document frequency (td-idf) to weight the corpus of words by the number of times a word appears in the document. Each document in the corpus is reduced to a fixed-length list of numbers, roughly corresponding to the frequency of appearance for a basic set of words within that document. This should help adjust for the fact that some words appear across all documents at a consistently high frequency. <br>

In order to implement this, we need several helper functions that turn our corpus of words (represented as a list of list of tuples) into a matrix (weighted frequencies) and then back into a corpus that can be inputted into the model.

In [ ]:
# takes a corpus and a number of words, and returns a matrix in which the element at row i and column j is the number of
# occurrences of word j in document i.
def corpus_to_mat(corpus, num_words):
    n = len(corpus)
    M = np.zeros((n, num_words))
    for i,doc in enumerate(corpus):
        for word,count in doc:
            M[i][word] = count
    return M

In [ ]:
%%time
#get noun corpus
nouncorpus = corpus_creator(nounwords,noun2id)
noun_train_corpus = [nouncorpus[i] for i in range(len(nouncorpus)) if sample_df['training'][i]==1]
noun_test_corpus = [nouncorpus[i] for i in range(len(nouncorpus)) if sample_df['training'][i]==0]

#get verb corpus
verbcorpus = corpus_creator(verbwords,verb2id)
verb_train_corpus = [verbcorpus[i] for i in range(len(verbcorpus)) if sample_df['training'][i]==1]
verb_test_corpus = [verbcorpus[i] for i in range(len(verbcorpus)) if sample_df['training'][i]==0]

#get adjective corpus
adjcorpus = corpus_creator(adjwords,adj2id)
adj_train_corpus = [adjcorpus[i] for i in range(len(adjcorpus)) if sample_df['training'][i]==1]
adj_test_corpus = [adjcorpus[i] for i in range(len(adjcorpus)) if sample_df['training'][i]==0]

#get foreign corpus
forcorpus = corpus_creator(forwords,for2id)
for_train_corpus = [forcorpus[i] for i in range(len(forcorpus)) if sample_df['training'][i]==1]
for_test_corpus = [forcorpus[i] for i in range(len(forcorpus)) if sample_df['training'][i]==0]

#get precedents corpus
preccorpus = corpus_creator(precedents_all,prec2id)
prec_train_corpus = [preccorpus[i] for i in range(len(preccorpus)) if sample_df['training'][i]==1]
prec_test_corpus = [preccorpus[i] for i in range(len(preccorpus)) if sample_df['training'][i]==0]

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer
#this function takes a training matrix of size n_documents_training*vocab_size and a test matrix
#of size n_documents_test*vocab_size. The function outputs the corresponding tfidf matrices.
#Note that we fit on the training data, and then apply that fit to the test data.
def tfidf_mat_creator(trainmatrix,testmatrix):
    tf_idf_transformer=TfidfTransformer()
    tfidf_fit = tf_idf_transformer.fit(trainmatrix)
    tfidf_train = tfidf_fit.transform(trainmatrix).toarray()
    tfidf_test = tfidf_fit.transform(testmatrix).toarray()
    return(tfidf_train,tfidf_test)

In [ ]:
noun_tfidf_mat_train,noun_tfidf_mat_test = tfidf_mat_creator(corpus_to_mat(noun_train_corpus, len(nounvocab)),
                                   corpus_to_mat(noun_test_corpus, len(nounvocab)))

In [ ]:
# takes a tfidf matrix and returns the corresponding matrix
def tfidf_to_corpus(tfidf_mat): #takes as input: matrix of size n_documents*vocabulary size
    tfidfcorpus = []
    i=0 #keep track of document you are on
    for doc in tfidf_mat: #for each case
        tfidfcorpus.append([])
        j=0
        for word in doc: #for each word in the vocabulary, append tuple (wordid,num_times_word_used)
            tfidfcorpus[i].append((j,tfidf_mat[i][j])) 
            j+=1
        i+=1
    return(tfidfcorpus) 

In [ ]:
tfidf_noun_corpus = tfidf_to_corpus(noun_tfidf_mat_train)

In [ ]:
%%time
# model with noun corpus (tf-idf weighted), 5 topics
lda2a = gensim.models.ldamodel.LdaModel(corpus=tfidf_noun_corpus, id2word=id2noun, num_topics=5, update_every=1, chunksize=200, passes=1)

In [ ]:
lda2a.print_topics()

In [ ]:
%%time
# model with noun corpus (tf-idf weighted), 10 topics
lda2b = gensim.models.ldamodel.LdaModel(corpus=tfidf_noun_corpus, id2word=id2noun, num_topics=10, update_every=1, chunksize=200, passes=1)

In [ ]:
lda2b.print_topics()

In [ ]:
for bow in corpus[0:100:5]:
    print lda2a.get_document_topics(bow)
    print " ".join([id2noun[e[0]] for e in bow])
    print "=========================================="

Although we can see that the tf-idf improved the performance of the model insofar as diluting the weights of words like "court" and "action," the model still classifies most cases as belonging to Topic 1, which is predicted by the all-purposes legal terms "court," "state," "act," "petitioner," "respondent," "statute," "case," "government," while  "district," and "u s." So, although it is possible to glean what sort of topics might be represented by the other categories (i.e. Topic 9 contains predictor words such as "search warrant" and "arrest" and therefore might be mapped to the issue area of privacy, while Topic 5 contains words like "tax," "property," and "revenue" and therefore might be mapped to the issue area of taxation), the model is biased towards assigning cases to the general topic of Topic 1.

### Third Pass (LSI)

Another model we might try before moving on to more supervised approaches (that might be more performant because they do not function under the assumption that there are no interactions among words). Latent Semantic Indexing (LSI) implements fast truncated SVD (Singular Value Decomposition), and performs well for text corpora much larger than RAM since only constant memory is needed. Since our text corpus is not terribly large, it's likely that LSI models will not output substantially better topic cluster than LDA, but it is worth trying this implementation.

In [ ]:
lsi1a = gensim.models.lsimodel.LsiModel(corpus=tfidf_noun_corpus, id2word=id2noun, num_topics=5, chunksize=200)

In [ ]:
lsi1a.print_topics()

In [ ]:
lsi1b = gensim.models.lsimodel.LsiModel(corpus=tfidf_noun_corpus, id2word=id2noun, num_topics=10, chunksize=200)

In [ ]:
lsi1b.print_topics()

As we predicted above, the LSI model did not output more discerning topics than the LDA model. Therefore, we can now move on to supervised models.